In [ ]:
MNIST 是 28×28 的灰度手写数字图，0-9 共 10 类。训练集 60000 张，测试集 10000 张。是深度学习界的"Hello World"。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# 固定随机种子，让 DataLoader 的 shuffle、模型初始化都可复现
# 不固定的话，每次重跑训练曲线会略有差异，对比时容易混淆
torch.manual_seed(42)

In [ ]:
# matplotlib 中文字体（按系统取一个能用的）
plt.rcParams['font.sans-serif'] = ['Heiti TC', 'PingFang SC', 'Songti SC', 'STHeiti', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# 数据预处理：转 tensor + 归一化
# - ToTensor: 把 PIL 图 (uint8, 0-255) 转成 float tensor, 范围 [0, 1], shape 由 (H, W, C) 变成 (C, H, W)
# - Normalize: 减均值除标准差，让像素分布零均值、单位方差
#   0.1307 / 0.3081 是 MNIST 训练集像素的均值/标准差（业界写死的常数）
#   为什么要这一步？后面归一化那节会展开讲——简单说，能让训练更稳、收敛更快
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

In [ ]:
# 第一次跑会自动从网上下载到 ./data/，之后走本地缓存
train_set = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_set = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)


# DataLoader 把数据集切成 batch
# - 训练集 shuffle=True: 每个 epoch 打乱顺序，避免模型记住数据出现的次序
# - 测试集 shuffle=False: 评估时顺序无所谓，关掉 shuffle 让结果可复现
train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False)

print(f'训练集: {len(train_set)} 张')
print(f'测试集: {len(test_set)} 张')
print(f'每张图: {train_set[0][0].shape} (通道 × 高 × 宽)')

In [ ]:
# 取 9 张样本看一下
fig, axes = plt.subplots(3, 3, figsize=(6, 6))
for i, ax in enumerate(axes.flat):
    img, label = train_set[i]
    # img 已经归一化过，反归一化回去显示（不然颜色不对）
    ax.imshow(img.squeeze() * 0.3081 + 0.1307, cmap='gray')
    ax.set_title(f'label: {label}', fontsize=12)
    ax.axis('off')
plt.suptitle('MNIST 训练集前 9 张', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
我们先不用 nn.Module，纯用 torch.tensor 把这 4 步写出来，看看底层在做什么。

### 网络结构

输入 (784)  --->  隐层 (128)  --->  输出 (10)
                 线性 + ReLU        线性

- 输入：28×28 = 784 维像素向量（把图片拍平）
- 隐层：128 个神经元，ReLU 激活
- 输出：10 维 logits，对应 10 个数字类别

In [ ]:
# ==========================================
# 手动初始化 4 个参数张量：W1, b1, W2, b2
# ==========================================
# 单独重跑这一段时，再固定一次种子，让权重初始化可复现
torch.manual_seed(42)

# requires_grad=True: 告诉 autograd"这个张量是参数，需要记录梯度"
# 没有这个标记，loss.backward() 算不出 p.grad
#
# (2.0 / n_in) ** 0.5 是 He 初始化的标准差，专门给 ReLU 网络用
# 经验值，先用着；为什么是 sqrt(2/n_in)，参数初始化那节再讲
W1 = (torch.randn(784, 128) * (2.0 / 784) ** 0.5).requires_grad_()
b1 = torch.zeros(128, requires_grad=True)

# 第 2 层: 128 -> 10 (输出 10 个类别的 logits)
W2 = (torch.randn(128, 10) * (2.0 / 128) ** 0.5).requires_grad_()
b2 = torch.zeros(10, requires_grad=True)

# 把 4 个参数放一个列表里，后面统一更新
params = [W1, b1, W2, b2]
lr = 0.1  # 学习率: 每次梯度下降迈多大一步。先拍 0.1，学习率与调度那节再细聊

print(f'网络共 {sum(p.numel() for p in params):,} 个参数')

In [ ]:
### 训练 5 个 epoch
- 一个 epoch = 把整个训练集 (60000 张) 完整过一遍
- 一个 batch = 128 张 (DataLoader 切的)，所以一个 epoch 大约 469 个 batch

In [ ]:
train_losses = []
test_accs = []

for epoch in range(5):
    running_loss = 0.0  # 累积本 epoch 所有 batch 的 loss × 样本数，最后求平均
    n_samples = 0
    
    for xb, yb in train_loader:
        # xb 形状 (128, 1, 28, 28), yb 形状 (128,)
        # 我们的网络只识别 1D 向量，所以把图像拍平成 (128, 784)
        x = xb.view(-1, 784)

        # ---- ① forward: 「线性 + 非线性」的组合 ----
        h = F.relu(x @ W1 + b1)      # 隐藏层: 内层线性变换，然后使用非线性函数 ReLU 激活
        logits = h @ W2 + b2         # 输出层: 只做线性变换，不加 softmax
                                     # (CrossEntropy 内部会做 log_softmax，自己加反而数值不稳)

        # ---- ② loss: 算预测和真值的差距 ----
        # cross_entropy 接收 logits (未归一化的分数) 和整数标签，自动做 softmax + 交叉熵
        loss = F.cross_entropy(logits, yb)

        # ---- ③ backward: 反向传播 ----
        # autograd 顺着前向那条计算图，从 loss 一路链式求导回每个 requires_grad 的张量
        # 求出来的梯度自动累加到 p.grad (所以下面要记得清零)
        loss.backward()

        # ---- ④ step: 梯度下降，沿梯度反方向更新参数 ----
        # 参数更新本身不需要建图 (不需要对"更新这一步"再求导)，所以包在 no_grad 里
        with torch.no_grad():
            for p in params:
                p -= lr * p.grad          # 沿梯度反方向迈 lr 这么大一步
                p.grad.zero_()            # 清零，防止下个 batch 的梯度叠加进来
        
        running_loss += loss.item() * yb.size(0)
        n_samples += yb.size(0)

    train_loss = running_loss / n_samples   # 整个 epoch 的平均 loss

    # ---- 评估: 测试集上跑一遍前向，统计准确率 (不算梯度，更快更省内存) ----
    correct = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            x = xb.view(-1, 784)
            logits = F.relu(x @ W1 + b1) @ W2 + b2
            correct += (logits.argmax(1) == yb).sum().item()
    test_acc = correct / len(test_set)

    train_losses.append(train_loss)
    test_accs.append(test_acc)
    print(f'epoch {epoch+1}/5  train_loss={train_loss:.4f}  test_acc={test_acc:.4f}')

In [ ]:
# 绘制训练 loss 和测试准确率曲线
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 训练 loss
axes[0].plot(range(1, 6), train_losses, marker='o')
axes[0].set_title('训练 loss')
axes[0].set_xlabel('epoch')
axes[0].set_ylabel('loss')
axes[0].grid(True)

# 测试准确率
axes[1].plot(range(1, 6), test_accs, marker='o', color='coral')
axes[1].set_title('测试准确率')
axes[1].set_xlabel('epoch')
axes[1].set_ylabel('accuracy')
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
## 3. 用 nn.Module 重写——同样的 4 步，更简洁

上面那段手写代码已经包含了 PyTorch 训练的全部本质。但实际工程里我们不会这么写——PyTorch 提供了：
- `nn.Module` 把"造网络 + 管理参数"打包好
- `torch.optim` 把"step + zero_grad"打包好
- `nn.Linear` 把 W @ x + b 打包好

下面这版代码和上面手写版结构上等价，每一行都和上面的 4 步严格对应：

In [ ]:
class SimpleMLP(nn.Module):
    """两层 MLP: 784 -> 128 -> 10"""
    
    # 继承 nn.Module 之后，PyTorch 自动管理:
    # - 把 self.fcN 里的参数收集起来 (model.parameters())
    # - .train() / .eval() 自动切换 Dropout、BN 等层的行为
    # - .to(device) 把所有参数搬到 GPU
    def __init__(self):
        super().__init__()
        # nn.Linear(in, out) 一行同时帮我们造好 W (out x in) 和 b (out)
        # 默认初始化是 Kaiming uniform——和上面手写的 He normal 不完全一样，但都是给 ReLU 用的
        self.fc1 = nn.Linear(784, 128)  # 等价于手写版 W1, b1
        self.fc2 = nn.Linear(128, 10)   # 等价于手写版 W2, b2

    def forward(self, x):
        x = x.view(-1, 784)            # 拍平成 (batch, 784)
        x = F.relu(self.fc1(x))        # 隐层 + ReLU
        return self.fc2(x)             # 输出 logits (不加 softmax, 原因同上)

In [ ]:
torch.manual_seed(42)
model = SimpleMLP()
optimizer = optim.SGD(model.parameters(), lr=0.1)  # SGD 内部就是手写版的 `p -= lr * p.grad`
loss_fn = nn.CrossEntropyLoss()                    # 等价于 F.cross_entropy

In [ ]:
train_losses_v2 = []
test_accs_v2 = []

for epoch in range(5):
    model.train()  # 进入"训练模式" (Dropout/BN 行为切换；这里没用到，但养成习惯)
    running_loss = 0.0
    n_samples = 0
    
    for xb, yb in train_loader:
        optimizer.zero_grad()        # ← 替代手写版的 `p.grad.zero_()`, 一行清空所有参数的梯度
        logits = model(xb)           # ---- ① forward —— 等价于 model.forward(xb)
        loss = loss_fn(logits, yb)   # ---- ② loss ——
        loss.backward()              # ---- ③ backward —— 还是 autograd 在做事
        optimizer.step()             # ---- ④ step —— 替代手写版的 `p -= lr * p.grad`
        
        running_loss += loss.item() * yb.size(0)
        n_samples += yb.size(0)
        
    model.eval()                     # 进入"评估模式"
    correct = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            correct += (model(xb).argmax(1) == yb).sum().item()
            
    train_losses_v2.append(running_loss / n_samples)
    test_accs_v2.append(correct / len(test_set))
    print(f'epoch {epoch+1}/5  train_loss={train_losses_v2[-1]:.4f}  test_acc={test_accs_v2[-1]:.4f}')

In [ ]:
## 4. 让模型识别图片
### 4.1 在测试集上预测 9 张

In [ ]:
# 随机抽 9 张测试图
import random
random.seed(7)  # 固定随机种子，每次抽到的都是同一组图（方便对照）
indices = random.sample(range(len(test_set)), 9)

model.eval()    # 推理模式
fig, axes = plt.subplots(3, 3, figsize=(7, 7))
with torch.no_grad():   # 推理时关掉梯度，省内存、加速
    for ax, idx in zip(axes.flat, indices):
        img, label = test_set[idx]
        # img 形状 (1, 28, 28), 模型期望 (batch, 1, 28, 28), 所以 unsqueeze(0) 添一维
        logits = model(img.unsqueeze(0))
        # softmax 把 logits 变成概率分布; [0] 是因为 batch=1
        probs = F.softmax(logits, dim=1)[0]
        pred = int(probs.argmax())     # 概率最大的那个类
        conf = float(probs[pred])      # 模型对自己预测的"信心"

        # 反归一化回原始像素范围再显示，不然颜色会偏 (前面 transform 里减过均值)
        ax.imshow(img.squeeze() * 0.3081 + 0.1307, cmap='gray')
        ok = (pred == label)
        ax.set_title(f'真值 {label}  预测 {pred}  ({conf:.0%})',
                     fontsize=10, color=('green' if ok else 'red'))
        ax.axis('off')
plt.suptitle('在测试集上的预测', fontsize=13, y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
### 4.2 现场画板 demo

下面是交互式画板，可以用鼠标在画板上画一个数字（0-9），按"识别"按钮，模型立刻识别它认为是几，以及对每个类别的置信度。

**第一次运行需要装 ipycanvas，装完后可能需要重启 kernel 才能让画板正常渲染——如果下面 cell 跑完看不到画板，重启 kernel 再从头运行一次。**

画的时候注意：
- 数字尽量画得大、居中、笔画粗 —— MNIST 训练时的数字都是这样的，离训练分布太远效果会差。
- 如果识别不准也别意外 —— 我们这只是个最简单的 2 层 MLP，没做任何工程优化。后面课程就是来"加 buff"的。

In [ ]:
# 第一次运行需要装画板控件 ipycanvas (已装可跳过)
# 装完之后可能需要重启 kernel 才能让画板正常渲染
%pip install ipycanvas pillow

from IPython.display import display
from ipycanvas import Canvas
from ipywidgets import Button, HBox, VBox, Output
from PIL import Image

# ---- 画板设置 ----
# 想改大改小就动 CANVAS_SIZE。建议保持为 28 的整数倍，缩放时比较干净。
CANVAS_SIZE = 560
LINE_WIDTH = max(8, CANVAS_SIZE // 14)  # 笔触粗细按画板大小自动缩放

canvas = Canvas(width=CANVAS_SIZE, height=CANVAS_SIZE, sync_image_data=True)
# 关键: JupyterLab 里 Canvas 默认会被 CSS 拉成全宽，必须显式约束 layout 大小
canvas.layout.width = f'{CANVAS_SIZE}px'

# 初始化画板样式（黑底白字，符合 MNIST 的格式）
canvas.fill_style = 'black'
canvas.fill_rect(0, 0, CANVAS_SIZE, CANVAS_SIZE)
canvas.stroke_style = 'white'
canvas.line_width = LINE_WIDTH

# ---- 鼠标交互逻辑 ----
brush = {'down': False, 'last': None}

def on_mouse_down(x, y):
    brush['down'] = True
    brush['last'] = (x, y)

def on_mouse_move(x, y):
    if brush['down']:
        last_x, last_y = brush['last']
        canvas.stroke_line(last_x, last_y, x, y)
        brush['last'] = (x, y)

def on_mouse_up(x, y):
    brush['down'] = False
    brush['last'] = None

canvas.on_mouse_down(on_mouse_down)
canvas.on_mouse_move(on_mouse_move)
canvas.on_mouse_up(on_mouse_up)

In [ ]:
# ---- 按钮和输出区 ----
output = Output()
btn_predict = Button(description='识别', button_style='primary', icon='check')
btn_clear = Button(description='清空', icon='trash')

def predict(_):
    output.clear_output()
    # 把 canvas 转成 28x28 灰度图
    arr = canvas.get_image_data()   # (H, W, 4) uint8 RGBA
    if arr is None:
        with output:
            print('画板还没准备好，等 1 秒再试，或者先在画板上随便画一笔')
        return
        
    img = Image.fromarray(arr).convert('L')          # 转灰度 (白笔->亮、黑底->暗)
    img = img.resize((28, 28), Image.LANCZOS)        # 缩到 28×28
    arr28 = np.array(img, dtype=np.float32) / 255.0  # [0, 1]
    
    # 用和训练时同样的归一化
    x = (torch.from_numpy(arr28) - 0.1307) / 0.3081
    
    model.eval()
    with torch.no_grad():
        logits = model(x.unsqueeze(0))
        probs = F.softmax(logits, dim=1)[0].numpy()
    pred = int(probs.argmax())
    
    with output:
        fig, ax = plt.subplots(1, 2, figsize=(9, 3))
        # 左: 模型实际看到的 28x28
        ax[0].imshow(arr28, cmap='gray')
        ax[0].set_title(f'模型看到的输入 (预测: {pred})')
        ax[0].axis('off')
        
        # 右: 预测概率分布
        ax[1].bar(range(10), probs * 100)
        ax[1].set_xticks(range(10))
        ax[1].set_title('预测概率 (%)')
        ax[1].set_ylim(0, 100)
        plt.show()

def clear(_):
    output.clear_output()
    canvas.fill_style = 'black'
    canvas.fill_rect(0, 0, CANVAS_SIZE, CANVAS_SIZE)

btn_predict.on_click(predict)
btn_clear.on_click(clear)

# ---- 布局整合 ----
container = VBox([
    canvas, 
    HBox([btn_predict, btn_clear]), 
    output
], layout={'width': f'{CANVAS_SIZE + 20}px', 'align_items': 'flex-start'})

display(container)